In [2]:
#This notebook contains the implementation of SFCMCP presented in "Semi-supervised Fuzzy Conformal Prediction for Data Stream Classiﬁcation". 
#It reproduces the main data-stream classification procedure described in Algorithms 1-3 of the manuscript.

import sklearn
from pandas import read_csv
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from statistics import mean
import numpy as np
import pandas as pd
import random
import scipy as sp
from skmultiflow.data import DataStream
from skmultiflow.bayes import NaiveBayes
import time

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DATA_PATH = "path/to/dataset.csv"
dataframe = read_csv(DATA_PATH)

dim=dataframe.shape
array = dataframe.values 
Y = array[:, dim[1]-1]
X = array[:, 0:dim[1]-1]

chunk_size=5000
class_set=np.unique(Y)
class_count=np.unique(Y).shape[0] # number of classess
d=X.shape[1] #dimension of data

################### SSFCM clustering parameters
m = 2 #fuzzy coefficinet
MAX_ITERS = 100 #maximum number of iterations for SSFCM
sigma = 0.00001 #progress threshold for SSFCM
lambda1=1 #weight of first term of loss function in range of (0,1]
lambda2=0.1 #weight of second term of loss function in range of (0,1]
Percentage=0.1 # percentage of labeled data in each chunk (0.05 or 0.1)
Mean_memb = 0.4
c = class_count
significance_level=0.90

def Unlabeling_data(X_train,Y_train,Percentage,chunk_size,class_count): 
    B=np.zeros(X_train.shape[0])
    labeled_count=int((Percentage*chunk_size)/class_count) # L per class
    for o in np.unique(Y_train):
        temp = np.argwhere(Y_train == o).flatten()        
        random.shuffle(temp)
        B[temp[:int(labeled_count)]] = 1
    return B

################## SSFCM clustering functions   
def initializeClusterCenters(X,Y,B,class_set):
    Centers = []
    Center_label = []
    for i in class_set:
        idx=np.intersect1d(np.argwhere(B==1),np.argwhere(Y==i))
        C=np.mean(X[idx],axis=0)
        Centers.append(C)
        Center_label.append(i)
        
    Centers=np.array(Centers)
    Center_label=np.array(Center_label)

    return Centers,Center_label


def ComputeF(Y,Center_label,n_current,c_current):
    F=np.zeros((n_current,c_current))
    g=0
    for i in Center_label:
        idx=np.argwhere(Y==i)
        F[idx,g]=1
        g=g+1
    return F

def ComputeS(So,Center_label,n_current,c_current):
    S=np.zeros((n_current,c_current))
    for i in np.unique(Center_label):
        idx=np.argwhere(Center_label==i).flatten()
        for ind in idx:
            S[:,ind]=So[:,int(i)]
    return S

# Updating membership matrix
def updateMembershipWeights(X,Umat,S,C,n_current,c_current): 
    denom = np.zeros(n_current)
    sigma = np.zeros(n_current)
    for i in range(c_current):
        dist = (X - C[i])**2
        dist = np.sum(dist, axis=1)
        dist = np.sqrt(dist)
        dist = np.maximum(dist, 1e-12)
        denom  = denom + np.power(1/dist,2/(m-1))
        sigma  = sigma + S[:,i]
    sigma=1-sigma 
    sigma=1+(lambda2*sigma)
    for i in range(c_current):
        dist = (X - C[i])**2
        dist = np.sum(dist, axis=1)
        dist = np.sqrt(dist)
        dist = np.maximum(dist, 1e-12)
        Umat[:,i] = np.divide(np.power(1/dist,2/(m-1)),denom)
        Umat[:,i]= sigma*Umat[:,i]
        
        sec_term=S[:,i] 
        Umat[:,i]=Umat[:,i]+(lambda2*sec_term)
        Umat[:,i]=Umat[:,i]*(np.divide(1,(1+lambda2)))
    return Umat

# Membership matrix initialization  (n*c)
def initializeMembershipWeights(n_current,c_current): 
    weight = np.random.dirichlet(np.ones(c_current),n_current)
    Umat = np.array(weight)
    return Umat

# Centroids Calculations   (c*d)   
def computeCentroids(X,Umat,S,c_current):
    C = []
    for i in range(c_current):
        U_i= np.power(Umat[:,i],m)
        mult=U_i.sum()
        sup  = np.power((Umat[:,i]-(S[:,i])),m)  
        sup_=sup.sum()
        denom=(lambda1*mult)+(lambda2*sup_)
        Cj = []
        for x in range(d):
            num = (lambda1*U_i)+(lambda2*sup)
            numerator = (X[:,x]*num).sum()
            c_val = numerator/denom
            Cj.append(c_val)
        C.append(Cj)
    C=np.array(C)
    return C


def Loss_calc(X,U,C,S,c_current): 
    J_calc=0
    term1=0
    term2=0
    for i in range(c_current):
        #term1
        dist = (X - C[i])**2
        dist = np.sum(dist, axis=1)
        U_i= np.power(U[:,i],m)
        mult=U_i*dist
        term1+= np.sum(mult,axis=0)
        
        #term2
        mult1 = np.power((U[:,i]-S[:,i]),m)  
        mult2= mult1*dist
        term2+= np.sum(mult2,axis=0)
    J_calc=(lambda1*term1)+(lambda2*term2)
    return J_calc

#Semi-supervised Fuzzy c_means Algorithm
def SSFCM(X,Y,n_current,C,S,c_current): 
    J_progress=[]
    U = initializeMembershipWeights(n_current,c_current)
    for z in range(MAX_ITERS):
        U = updateMembershipWeights(X,U,S,C,n_current,c_current)
        C = computeCentroids(X,U,S,c_current)
        J_calc=Loss_calc(X,U,C,S,c_current)
        J_progress.append(J_calc)
        num_iters=z+1
        if z>0:
            if (np.abs(J_progress[z-1]-J_progress[z]))<=sigma: # if the Loss progress is not sufficient
                break
               
    return U,C,J_progress,num_iters

def accuracy(Y,U,Center_label,n_current):  
    Y_=np.zeros(n_current)
    idx=np.argmax(U,axis=1)
    for j in range(n_current):
        Y_[j]=Center_label[idx[j]]
    acc=accuracy_score(Y,Y_)
    return acc

def summarize(X,U,C,S,c_current):
    
    LC=np.zeros(c_current)
    SM=np.zeros(c_current)
    LS_M=[]
    LS_LC=[]
        
    for i in range(c_current):
        U_i= np.power(U[:,i],m)
        SM[i]= np.sum(U_i,axis=0) #SM
        
        temp = np.power((U[:,i]-S[:,i]),m) 
        LC[i]= np.sum(temp,axis=0) #LC
        
        lsm = []
        lslc=[]
        for x in range(d):
            u_val = ( X[:,x] * U_i).sum()
            lsm.append(u_val)
            LC_val = ( X[:,x] * temp).sum()
            lslc.append(LC_val)

        LS_M.append(lsm) #LS_M
        LS_LC.append(lslc) #LS_LC
    LS_M=np.array(LS_M)
    LS_LC=np.array(LS_LC)    
    return LS_M,LS_LC,SM,LC

def New_model_construction(SM,LS_M,LC,LS_LC,C,C_label,MC_weights,Xo,Bo,Yo,n_MC):
    
    # Selecting accurate new centers by running SSFCM on labeled set for which a drift was detected
    [Centers,Center_label] =initializeClusterCenters(Xo,Yo,Bo,np.unique(Yo))
    c_current=np.unique(Yo).shape[0]
    Fo=ComputeF(Yo,Center_label,Xo.shape[0],c_current)
    [Uo,Co,J_progress_,num_iters_]=SSFCM(Xo,Yo,Yo.shape[0],Centers,Fo,c_current)
    
    # Adding new centers to Centers list
    C=np.concatenate((C,Co),axis=0)
    C_label=np.concatenate((C_label,Center_label),axis=0)
    n_MC=n_MC+c_current
    MC_weights=np.concatenate((MC_weights,np.ones(c_current)),axis=0)
    
    # Constructing U between new chunk's labeled set and all the micro-clusters
    [F_labeled,U_labeled]=Predict(Xo,Yo,Bo,C,C_label,Xo.shape[0],n_MC,Fo,"L")
    
    [LS_Mo,LS_LCo,SMo,LCo]=summarize(Xo,Uo,Co,Fo,c_current)
    
    # Updating the model
    LC=np.concatenate((LC, LCo), axis=0)
    SM=np.concatenate((SM, SMo), axis=0)
    LS_M=np.concatenate((LS_M, LS_Mo), axis=0)
    LS_LC=np.concatenate((LS_LC, LS_LCo), axis=0)

    
    return C,C_label,SM,LS_M,LC,LS_LC,MC_weights,n_MC,U_labeled

################## Model initialization 
def Model_Initialization(X,Y,B,n_current,s):
    [Centers,Center_label] =initializeClusterCenters(X,Y,B,np.unique(Y))
    c_current=np.unique(Y).shape[0]
    F=ComputeF(Y,Center_label,n_current,c_current)
    S=ComputeS(s,Center_label,n_current,c_current)
    for x in range(X.shape[0]):
        if B[x]==1:
            S[x,:]=F[x,:]
    [U,C,J_progress,num_iters]=SSFCM(X,Y,Y.shape[0],Centers,S,c_current)
    return U,C,Center_label,F,c_current

def Predict(X,Y,B,C,Center_label,n,n_MC,s,s_type):
    if s_type=="L":
        S = ComputeF(Y,Center_label,n,n_MC)
    else:
        S=ComputeS(s,Center_label,n,n_MC)
    U = np.zeros([X.shape[0],n_MC])
    U = updateMembershipWeights(X,U,S,C,n,n_MC)
    return S,U

def update_statistics(C,LS_M,LS_LC,SM,LC,cls,Ux,x,Sx):

    U_i= np.power(Ux[cls],m)
    SM[cls] = SM[cls] + U_i #SM
    
    LS_M[cls] = LS_M[cls] + (x*U_i) #LS_M
    
    temp=np.power((Ux[cls]-Sx[cls]),m) 
    LS_LC[cls] = LS_LC[cls] + (x*temp) #LS_LC
    LC[cls] =LC[cls]+temp #LC
    
    C[cls]=(lambda2*LS_LC[cls])+(lambda1*LS_M[cls])
    denom=(lambda2*LC[cls])+(lambda1*SM[cls])
    C[cls]=np.divide(C[cls],denom) #Center
    return C,LS_M,LS_LC,SM,LC

def Train(C,Class_label,LS_M,LS_LC,SM,LC,Ux,Xx,Yx,Sx,status,Memb_th):
    if status=="unlabeled":
        for x in range (Xx.shape[0]):
            if np.max(Ux[x,:])>=Memb_th:
                idx=np.argwhere(Ux[x,:]>=Mean_memb).flatten()
                for cls in idx:
                    [C,LS_M,LS_LC,SM,LC]=update_statistics(C,LS_M,LS_LC,SM,LC,cls,Ux[x,:],Xx[x,:],Sx[x,:])
    else:
        for x in range (Xx.shape[0]):
            if (Yx[x]==Center_label[np.argmax(Ux[x,:])]):
                idx=np.argwhere(Ux[x,:]==np.max(Ux[x,:])).flatten()
                for cls in idx:
                    [C,LS_M,LS_LC,SM,LC]=update_statistics(C,LS_M,LS_LC,SM,LC,cls,Ux[x,:],Xx[x,:],Sx[x,:])
    return C,LS_M,LS_LC,SM,LC

# Conformal Prediction
def Prediction_by_CP(U_labeled,Y_labeled,U_unlabeled,Center_label): 
    row=U_unlabeled.shape[0]
    col=np.unique(Center_label).shape[0]
    p_values=np.zeros([row,col])
    labels = np.ones((row,col),dtype=bool)
    class_set = np.unique(Center_label)
    alphas = NCM(U_labeled,Y_labeled,Center_label,class_set,1)
    for elem in range(row):
        cp=[]
        for o in class_set:
            a_test = NCM(U_unlabeled[elem,:],o,Center_label,class_set,2)
            idx = np.argwhere(Y_labeled == o).flatten()
            temp=alphas[idx]
            p=len(temp[temp>=a_test])
            if idx.shape[0]==0:
                s=0
            else:
                s=p/idx.shape[0]
            cp.append(s)
            if s<significance_level:
                labels[elem,int(o)]=False
        p_values[elem,:]=np.array(cp)
    return p_values,labels

# Calculating non-confomity (NCM) scores
def NCM(U,Y,Center_label,class_set,t):
    if t==1:
        P=np.zeros([U.shape[0],1])
        for x in range(U.shape[0]):
            idx = np.argwhere(Center_label == Y[x]).flatten()
            Uselect=U[x,idx]
            if idx.shape[0]>1:
                P[x] = np.sum(Uselect)
            else:
                P[x] = Uselect

        alpha=1-P
    else: #t==2
        idx = np.argwhere(Center_label == int(Y)).flatten()
        P=np.sum(U[idx])
        alpha=1-P
    return alpha
    


##################################################### Offline phase 
time_list=[]                      
start_time=time.time()  

#Processing the first data chunk
stream=DataStream(X,Y,target_idx=0, n_targets=class_count, cat_features=None, name=None, allow_nan=False)
X_chunk1, Y_chunk1 = stream.next_sample(chunk_size)
X_train, X_test, Y_train, Y_test = train_test_split(X_chunk1, Y_chunk1, test_size=0.2,random_state=RANDOM_STATE)
n=X_train.shape[0]
B=Unlabeling_data(X_train,Y_train,Percentage,X_train.shape[0],np.unique(Y_train).shape[0])
classifier = NaiveBayes() 
classifier.fit(X_train[B==1],Y_train[B==1])
S = classifier.predict_proba(X_train)
[U,C,Center_label,F,c_current]=Model_Initialization(X_train,Y_train,B,n,S)
[LS_M,LS_LC,SM,LC]=summarize(X_train,U,C,S,c_current)

S = classifier.predict_proba(X_test)
[ff,U_test]=Predict(X_test,Y_test,np.zeros(X_test.shape[0]),C,Center_label,X_test.shape[0],c_current,S,"U")
acc=accuracy(Y_test,U_test,Center_label,X_test.shape[0])
print(acc)
A=[]
A.append(acc)

#Initialization parameters
max_MC=30 # maximum number of micro-clusters per class
n_MC=c_current
Memb_th=0.7
MC_weights=np.ones(c_current)

##################################################### Online phase
# Processing the Second data chunk
X_chunk2, Y_chunk2 = stream.next_sample(chunk_size)
X_train, X_test, Y_train, Y_test = train_test_split(X_chunk2, Y_chunk2, test_size=0.2,random_state=RANDOM_STATE)
B=Unlabeling_data(X_train,Y_train,Percentage,X_train.shape[0],np.unique(Y_train).shape[0])
classifier.partial_fit(X_train[B==1],Y_train[B==1])
S = classifier.predict_proba(X_train[B==1])
[S,U]=Predict(X_train[B==1],Y_train[B==1],B[B==1],C,Center_label,(X_train[B==1]).shape[0],n_MC,S,"L")
[C,LS_M,LS_LC,SM,LC]=Train(C,Center_label,LS_M,LS_LC,SM,LC,U,X_train[B==1],Y_train[B==1],S,"labeled",Memb_th)


S = classifier.predict_proba(X_test)
[ff,U_test]=Predict(X_test,Y_test,np.zeros(X_test.shape[0]),C,Center_label,X_test.shape[0],n_MC,S,"U")
acc=accuracy(Y_test,U_test,Center_label,X_test.shape[0])
A.append(acc)
print(acc)


a_file = open("Kolmogorov.txt", "w") 
X_previous=X_train
Y_previous=Y_train
X_test_previous=X_test
Y_test_previous=Y_test
U_labeled=U
F_labeled=F
B_=B
num_samples = X.shape[0]-(2*chunk_size) 
n_samples = 0
i=3
flag=0
Kolmogorov=[]
time_list.append(time.time()-start_time) 
while n_samples < num_samples  and stream.has_more_samples():
    S_ = classifier.predict_proba(X_previous[B_==0])
    [S_,U_]=Predict(X_previous[B_==0],Y_previous[B_==0],B_[B_==0],C,Center_label,(X_previous[B_==0]).shape[0],n_MC,S_,"U")
    p_values,labels = Prediction_by_CP(U_labeled,Y_previous[B_==1],U_,Center_label) #Ui-1

    X_current, Y_current = stream.next_sample(chunk_size)
    X_train, X_test, Y_train, Y_test = train_test_split(X_current, Y_current, test_size=0.2,random_state=RANDOM_STATE)
    B=Unlabeling_data(X_train,Y_train,Percentage,X_train.shape[0],np.unique(Y_train).shape[0])
    S = classifier.predict_proba(X_train[B==0])
    [S,U]=Predict(X_train[B==0],Y_train[B==0],B[B==0],C,Center_label,(X_train[B==0]).shape[0],n_MC,S,"U") #Ui 
    p_values1,labels1 = Prediction_by_CP(U_labeled,Y_previous[B_==1],U,Center_label)
    
    [C,LS_M,LS_LC,SM,LC]=Train(C,Center_label,LS_M,LS_LC,SM,LC,U_,X_previous[B_==0],Y_previous[B_==0],S_,"unlabeled",Memb_th)
    
    
    S = classifier.predict_proba(X_test_previous)
    [ff,U_test]=Predict(X_test_previous,Y_test_previous,np.zeros(X_test_previous.shape[0]),C,Center_label,X_test_previous.shape[0],n_MC,S,"U")
    p_values_t,labels_t = Prediction_by_CP(U_labeled,Y_previous[B_==1],U_test,Center_label)
    acc_t=accuracy(Y_test_previous,p_values_t,Center_label,X_test_previous.shape[0])
    acc=accuracy(Y_test_previous,U_test,Center_label,X_test_previous.shape[0])
    A.append(max(acc,acc_t))
    print(acc)  
    
    if X_current.shape[0]>=chunk_size: #if not last chunk
        kst=[]
        class_set=np.unique(Y_train)
        for h in class_set:
            val=sp.stats.ks_2samp(p_values[:,int(h)], p_values1[:,int(h)])
            kst.append(val[1])
        min_kst=min(kst)
        Kolmogorov.append(min_kst)
        print("chunk"+str(i-1)+" --> "+str(i))
        print("kolmogorov smirnov:    "+str(min_kst))
        if min_kst<0.05:  #if drift
            classifier = NaiveBayes()
            classifier.fit(X_train[B==1],Y_train[B==1])
            [C,Center_label,SM,LS_M,LC,LS_LC,MC_weights,n_MC,U_labeled]=New_model_construction(SM,LS_M,LC,LS_LC,C,Center_label,MC_weights,X_train[B==1],np.ones((Y_train[B==1]).shape[0]),Y_train[B==1],n_MC)
        else: # if no drift
            classifier.partial_fit(X_train[B==1],Y_train[B==1])
            S = classifier.predict_proba(X_train[B==1])
            [S_labeled,U_labeled]=Predict(X_train[B==1],Y_train[B==1],B[B==1],C,Center_label,(X_train[B==1]).shape[0],n_MC,S,"L")
            [C,LS_M,LS_LC,SM,LC]=Train(C,Center_label,LS_M,LS_LC,SM,LC,U_labeled,X_train[B==1],Y_train[B==1],S_labeled,"labeled",Memb_th)
    else: #if last chunk
        S = classifier.predict_proba(X_train[B==1])
        [S_labeled,U_labeled]=Predict(X_train[B==1],Y_train[B==1],B[B==1],C,Center_label,(X_train[B==1]).shape[0],n_MC,S,"L")
        [C,LS_M,LS_LC,SM,LC]=Train(C,Center_label,LS_M,LS_LC,SM,LC,U_labeled,X_train[B==1],Y_train[B==1],S_labeled,"labeled",Memb_th)
        
    X_previous=X_train
    Y_previous=Y_train
    B_=B
    X_test_previous=X_test
    Y_test_previous=Y_test
    
    time_list.append(time.time()-start_time)
    n_samples += chunk_size
    i+=1
    
np.savetxt("SSFCM_time.txt",np.array(time_list),fmt="%.6f") 
ss_file = open("SFCMCP.txt", "w")
np.savetxt(ss_file, np.array(A), fmt='%.4f', delimiter='\t\t', newline='\n')
ss_file.close()
np.savetxt(a_file, Kolmogorov, fmt='%.200f', delimiter='\t\t', newline='\n')
a_file.close()
Accuracy=np.mean(np.array(A),axis=0) 
print("****Overall accuracy*****")
print("Acc approach:   "+str(Accuracy))

The watermark extension is already loaded. To reload it, use:
  %reload_ext watermark
Python implementation: CPython
Python version       : 3.8.3
IPython version      : 7.16.1

numpy       : 1.23.5
pandas      : 1.5.1
scipy       : 1.10.1
skmultiflow : 0.5.3
scikit-learn: 1.3.2



'\nDATA_PATH = "path/to/dataset.csv"\ndataframe = read_csv(DATA_PATH)\n\ndim=dataframe.shape\narray = dataframe.values \nY = array[:, dim[1]-1]\nX = array[:, 0:dim[1]-1]\n\nchunk_size=5000\nclass_set=np.unique(Y)\nclass_count=np.unique(Y).shape[0] # number of classess\nd=X.shape[1] #dimension of data\n\n################### SSFCM clustering parameters\nm = 2 #fuzzy coefficinet\nMAX_ITERS = 100 #maximum number of iterations for SSFCM\nsigma = 0.00001 #progress threshold for SSFCM\nlambda1=1 #weight of first term of loss function in range of (0,1]\nlambda2=0.1 #weight of second term of loss function in range of (0,1]\nPercentage=0.1 # percentage of labeled data in each chunk (0.05 or 0.1)\nMean_memb = 0.4\nc = class_count\nsignificance_level=0.90\n\ndef Unlabeling_data(X_train,Y_train,Percentage,chunk_size,class_count): \n    B=np.zeros(X_train.shape[0])\n    labeled_count=int((Percentage*chunk_size)/class_count) # L per class\n    for o in np.unique(Y_train):\n        temp = np.argwhere(